<a href="https://colab.research.google.com/github/ishwaryaponnusamy56-ctrl/doc_genie/blob/main/Doc_genie_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries (Run only in Jupyter/Colab)
!pip install -q gradio transformers torch black autopep8 reportlab requests

# Imports
import gradio as gr
from datetime import datetime
from pathlib import Path
import json
import re
import ast
import inspect
import requests
import urllib.parse

# ReportLab imports
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    Table,
    TableStyle,
    Preformatted
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 18.9 MB/s eta 0:00:00


In [ ]:
class DocGenieAnalyzer:
    """Analyze Python functions and generate professional docstrings"""

    @staticmethod
    def extract_function_signature(code):
        """Extract function name, parameters, return type from code"""
        try:
            tree = ast.parse(code)
            func_def = tree.body[0]

            if not isinstance(func_def, ast.FunctionDef):
                return None, None

            func_name = func_def.name
            params = []

            # Extract parameters
            for arg in func_def.args.args:
                param_type = "Any"
                if arg.annotation:
                    param_type = (
                        ast.unparse(arg.annotation)
                        if hasattr(ast, "unparse")
                        else "Any"
                    )

                params.append({
                    "name": arg.arg,
                    "type": param_type,
                    "default": None
                })

            # Extract default values
            for i, default in enumerate(func_def.args.defaults):
                idx = len(params) - len(func_def.args.defaults) + i
                if idx >= 0:
                    params[idx]["default"] = (
                        ast.unparse(default)
                        if hasattr(ast, "unparse")
                        else str(default)
                    )

            # Extract return type
            return_type = "Any"
            if func_def.returns:
                return_type = (
                    ast.unparse(func_def.returns)
                    if hasattr(ast, "unparse")
                    else "Any"
                )

            return {
                "name": func_name,
                "params": params,
                "return_type": return_type
            }, func_def

        except Exception:
            return None, None

    @staticmethod
    def analyze_function_logic(func_def, code):
        """Deep analysis of function body to understand actual behavior"""
        try:
            analysis = {
                "has_loops": False,
                "has_conditions": False,
                "has_return": False,
                "loop_types": [],
                "condition_types": [],
                "operations": [],
                "return_types": [],
                "variables_used": set(),
                "function_calls": [],
                "description": ""
            }

            # Walk through all AST nodes in the function
            for node in ast.walk(func_def):

                if isinstance(node, ast.For):
                    analysis["has_loops"] = True
                    analysis["loop_types"].append("for")

                elif isinstance(node, ast.While):
                    analysis["has_loops"] = True
                    analysis["loop_types"].append("while")

                elif isinstance(node, ast.If):
                    analysis["has_conditions"] = True
                    analysis["condition_types"].append("if")

                elif isinstance(node, ast.Return):
                    analysis["has_return"] = True
                    if node.value:
                        analysis["return_types"].append(
                            ast.unparse(node.value) if hasattr(ast, "unparse") else "value"
                        )

                elif isinstance(node, ast.BinOp):
                    op = node.op
                    if isinstance(op, ast.Mult):
                        analysis["operations"].append("multiplication")
                    elif isinstance(op, ast.Add):
                        analysis["operations"].append("addition")
                    elif isinstance(op, ast.Sub):
                        analysis["operations"].append("subtraction")
                    elif isinstance(op, ast.Div):
                        analysis["operations"].append("division")

                elif isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        analysis["function_calls"].append(node.func.id)

                elif isinstance(node, ast.Name):
                    analysis["variables_used"].add(node.id)

            # Build detailed description
            description_parts = []

            if func_def.name:
                description_parts.append(f"Executes {func_def.name} operation")

            if analysis["has_conditions"]:
                description_parts.append("with conditional logic")

            if analysis["has_loops"]:
                description_parts.append("and iterates through values")

            if "multiplication" in analysis["operations"] and "subtraction" in analysis["operations"]:
                description_parts.append("performing mathematical calculations")
            elif "multiplication" in analysis["operations"]:
                description_parts.append("using multiplication operations")
            elif "addition" in analysis["operations"]:
                description_parts.append("using addition operations")
            elif "division" in analysis["operations"]:
                description_parts.append("using division operations")

            if analysis["return_types"]:
                description_parts.append("and returns the computed result")

            analysis["description"] = " ".join(description_parts) + "."

            if len(analysis["description"]) < 20:
                analysis["description"] = "Performs computation based on input parameters."

            # Convert set to list for JSON/output compatibility
            analysis["variables_used"] = list(analysis["variables_used"])

            return analysis

        except Exception:
            return {
                "has_loops": False,
                "has_conditions": False,
                "has_return": False,
                "loop_types": [],
                "condition_types": [],
                "operations": [],
                "return_types": [],
                "variables_used": [],
                "function_calls": [],
                "description": "Unable to analyze function logic."
            }

    @staticmethod
    def generate_google_docstring(signature, analysis, code):
        """Generate accurate Google-style docstring"""
        summary = analysis.get("description", "Processes input and returns result.")

        # Start docstring
        docstring = f'"""{summary}\n'

        # Parameters section
        if signature.get("params"):
            docstring += "\nArgs:\n"
            for param in signature["params"]:
                default_str = f" (default: {param['default']})" if param.get("default") else ""
                docstring += (
                    f"    {param['name']} ({param['type']}): "
                    f"Parameter for controlling {param['name']} behavior{default_str}.\n"
                )

        # Return section
        docstring += f"\nReturns:\n    {signature['return_type']}: The result of the computation.\n"
        docstring += '"""\n'

        return docstring

    @staticmethod
    def generate_numpy_docstring(signature, analysis, code):
        """Generate accurate NumPy-style docstring"""
        summary = analysis.get("description", "Processes input and returns result.")

        # Start docstring
        docstring = f'"""{summary}\n'

        # Parameters section
        if signature.get("params"):
            docstring += "\nParameters\n----------\n"
            for param in signature["params"]:
                default_str = f", default: {param['default']}" if param.get("default") else ""
                docstring += f"    {param['name']} : {param['type']}{default_str}\n"
                docstring += f"        Parameter for controlling {param['name']} behavior.\n"

        # Returns section
        docstring += f"\nReturns\n-------\n    {signature['return_type']}\n"
        docstring += "        The result of the computation.\n"
        docstring += '"""\n'

        return docstring

In [ ]:
from datetime import datetime

# Global history for generated docstrings
generation_history = []
analyzer = DocGenieAnalyzer()


def generate_docstring(code, style="google"):
    """Generate docstring for Python function with accurate analysis"""
    if not code.strip():
        return None, "❌ Empty code provided!"

    try:
        signature, func_def = analyzer.extract_function_signature(code)
        if not signature:
            return None, "❌ Invalid Python function code!"

        # Deep analysis of the function
        analysis = analyzer.analyze_function_logic(func_def, code)

        # Generate docstring based on style
        if style.lower() == "google":
            docstring = analyzer.generate_google_docstring(signature, analysis, code)
        else:
            docstring = analyzer.generate_numpy_docstring(signature, analysis, code)

        # Extract function definition line
        code_lines = code.split("\n")
        func_line = code_lines[0]

        # Build final function with docstring
        final_code = func_line + "\n" + docstring
        for line in code_lines[1:]:
            final_code += line + "\n"

        # Save generation history
        generation_history.append({
            "code": code,
            "docstring": docstring,
            "style": style,
            "timestamp": datetime.now().isoformat(),
            "analysis": analysis
        })

        return final_code, f"✅ {style.upper()} Docstring Generated!"

    except Exception as e:
        return None, f"❌ Error: {str(e)}"


def process_upload(file):
    """Process uploaded Python file"""
    if not file:
        return None, "❌ No file uploaded!"

    try:
        content = file.read().decode("utf-8")
        return content, "✅ File loaded successfully!"
    except Exception:
        return None, "❌ Failed to read file!"


def clear_history():
    """Clear generation history"""
    global generation_history
    generation_history = []
    return "✅ History cleared!"


def download_pdf():
    """Download all generated docstrings as PDF"""
    if not generation_history:
        return "❌ No documentation generated!", None

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"doc_genie_{timestamp}.pdf"

    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted
    from reportlab.lib.styles import getSampleStyleSheet

    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    flowables = []

    for record in generation_history:
        flowables.append(Paragraph(f"Function: {record['code'].splitlines()[0]}", styles["Heading3"]))
        flowables.append(Spacer(1, 0.1 * inch))
        flowables.append(Preformatted(record["docstring"], styles["Code"]))
        flowables.append(Spacer(1, 0.3 * inch))

    doc.build(flowables)

    return f"✅ PDF generated: {filename}", filename

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.pagesizes import A4
from datetime import datetime

def download_pdf_a4(generation_history):
    """Generate PDF report using A4 page size"""
    try:
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        filename = f"doc_genie_{timestamp}.pdf"

        doc = SimpleDocTemplate(
            filename,
            pagesize=A4,
            topMargin=0.5 * inch,
            bottomMargin=0.5 * inch
        )
        styles = getSampleStyleSheet()
        story = []

        title_style = ParagraphStyle(
            'Title',
            parent=styles['Heading1'],
            fontSize=24,
            textColor='#FF6B6B',
            spaceAfter=20,
            alignment=1  # Center
        )

        code_style = ParagraphStyle(
            'Code',
            parent=styles['Normal'],
            fontSize=9,
            fontName='Courier',
            backColor='#F0F0F0',
            borderColor='#CCCCCC',
            borderWidth=1,
            borderPadding=6
        )

        # Add title and generation timestamp
        story.append(Paragraph("📚 Doc-Genie Documentation Report", title_style))
        story.append(Paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles['Normal']))
        story.append(Spacer(1, 0.3 * inch))

        # Add each generated documentation
        for i, gen in enumerate(generation_history, 1):
            story.append(Paragraph(f"<b>Documentation #{i}</b>", styles['Heading2']))
            story.append(Paragraph(f"<b>Style:</b> {gen['style'].upper()}", styles['Normal']))
            story.append(Spacer(1, 0.1 * inch))

            story.append(Paragraph("<b>Original Code:</b>", styles['Normal']))
            story.append(Preformatted(gen['code'], code_style))
            story.append(Spacer(1, 0.15 * inch))

            story.append(Paragraph("<b>Generated Docstring:</b>", styles['Normal']))
            story.append(Preformatted(gen['docstring'], code_style))
            story.append(Spacer(1, 0.3 * inch))

            # Add page break after every 2 documentations
            if i % 2 == 0 and i != len(generation_history):
                story.append(PageBreak())

        # Add total count
        story.append(Spacer(1, 0.3 * inch))
        story.append(Paragraph(f"Total Documentations: {len(generation_history)}", styles['Normal']))

        # Build PDF
        doc.build(story)

        return f"✅ PDF Downloaded: {filename}", filename

    except Exception as e:
        return f"❌ Error: {str(e)}", None

In [ ]:
from datetime import datetime
from pathlib import Path
import urllib.parse as urlparse

def download_txt():
    """Download all generated docstrings as TXT"""
    if not generation_history:
        return "❌ No documentation generated!", None

    content = f"Doc-Genie Documentation Report\nGenerated: {datetime.now()}\n{'='*80}\n\n"

    for i, gen in enumerate(generation_history, 1):
        content += f"Documentation #{i}\n"
        content += f"Style: {gen['style'].upper()}\n"
        content += f"Generated: {gen['timestamp']}\n"
        content += f"{'-'*80}\n"
        content += f"ORIGINAL CODE:\n{gen['code']}\n\n"
        content += f"GENERATED DOCSTRING:\n{gen['docstring']}\n\n"
        content += f"{'='*80}\n\n"

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"doc_genie_{timestamp}.txt"

    try:
        Path(filename).write_text(content)
        return f"✅ TXT Downloaded: {filename}", filename
    except Exception as e:
        return f"❌ Error: {str(e)}", None


def generate_whatsapp_link():
    """Generate WhatsApp share link"""
    if not generation_history:
        return "❌ No documentation to share!"

    message = "📚 *Doc-Genie Documentation*\n\n"
    for i, gen in enumerate(generation_history[-3:], 1):
        message += f"*Documentation #{i}*\n"
        message += f"Style: {gen['style'].upper()}\n"
        message += f"Code: {gen['code'][:60]}...\n\n"

    encoded = urlparse.quote(message)
    link = f"https://wa.me/?text={encoded}"

    return f"✅ WhatsApp Link:\n{link}\n\n Share with team!"


def generate_facebook_link():
    """Generate Facebook share link"""
    if not generation_history:
        return "❌ No documentation to share!"

    message = f"I generated {len(generation_history)} professional Python function docstrings with Doc-Genie! Check it out!"

    encoded = urlparse.quote(message)
    link = f"https://www.facebook.com/sharer/sharer.php?u={encoded}"

    return f"✅ Facebook Link:\n{link}\n\n Share on Facebook!"

In [ ]:
import gradio as gr

print("🚀 Loading Doc-Genie Improved Version...")

# Helper function for file upload processing, defined globally
def process_and_display(file):
    if file:
        content, status = process_upload(file)
        return content, status
    return "", "No file"

with gr.Blocks(
    title="Doc-Genie - Python Documentation Generator",
    theme=gr.themes.Soft(),
    css="""
    body { background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); }
    .gradio-container { background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); }
    .button-primary { background: linear-gradient(135deg, #FF6B6B 0%, #FF8E72 100%); }
    .button-secondary { background: linear-gradient(135deg, #4ECDC4 0%, #44A08D 100%); }
    .code-block { background: #0f3460; border: 2px solid #4ECDC4; border-radius: 8px; padding: 15px; }
    """
) as demo:

    gr.Markdown("""
    # 📚 Doc-Genie v2.0
    ## Professional Python Function Docstring Generator

    Transform your Python functions with AI-powered, accurate documentation!
    """)

    with gr.Row():
        with gr.Column(scale=3):
            # Code Input Area
            gr.Markdown("### 📝 Input Function Code")

            code_input = gr.Code(
                language="python",
                label="Python Function",
                lines=10,
                value=(
                    "def calculate_factorial(n):\n"
                    "    if n == 0:\n"
                    "        return 1\n"
                    "    return n * calculate_factorial(n-1)"
                )
            )

            # File Upload
            file_upload = gr.File(label="📤 Or Upload .py File", file_types=['.py'])
            upload_status = gr.Textbox(label="Upload Status", interactive=False, lines=1)

            file_upload.change(process_and_display, inputs=[file_upload], outputs=[code_input, upload_status])

            # Style Selection and Generate Button
            with gr.Row():
                docstring_style = gr.Radio(["google", "numpy"], value="google", label="🎨 Docstring Style")
                generate_btn = gr.Button("✨ Generate Docstring", size="lg", variant="primary", scale=2)

            # Output Area
            gr.Markdown("### 📄 Generated Output")

            code_output = gr.Code(
                language="python",
                label="Function with Docstring",
                lines=15,
                interactive=False
            )

            gen_status = gr.Textbox(label="Status", interactive=False, lines=1)

            generate_btn.click(
                generate_docstring,
                inputs=[code_input, docstring_style],
                outputs=[code_output, gen_status]
            )

        with gr.Column(scale=1):
            gr.Markdown("### 🎮 Controls")

            clear_btn = gr.Button("🗑️ Clear History", variant="stop", size="sm")
            clear_btn.click(clear_history, outputs=[upload_status])

            gr.Markdown("---")
            gr.Markdown("### 📥 Export")

            pdf_btn = gr.Button("📄 Download PDF", size="sm", variant="primary")
            txt_btn = gr.Button("📋 Download TXT", size="sm", variant="primary")

            pdf_file = gr.File(label="PDF File", visible=True)
            txt_file = gr.File(label="TXT File", visible=True)
            export_status = gr.Textbox(label="Export Status", interactive=False, lines=2)

            pdf_btn.click(download_pdf, outputs=[export_status, pdf_file])
            txt_btn.click(download_txt, outputs=[export_status, txt_file])

🚀 Loading Doc-Genie Improved Version...


/tmp/ipykernel_580/340187024.py:12: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_580/340187024.py:12: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


In [ ]:
share_link_whatsapp = generate_whatsapp_link()
print(share_link_whatsapp)

✅ WhatsApp Link:
https://wa.me/?text=%F0%9F%93%9A%20%2ADoc-Genie%20Documentation%2A%0A%0A%2ADocumentation%20%231%2A%0AStyle%3A%20GOOGLE%0ACode%3A%20def%20calculate_factorial%28n%29%3A%0A%20%20%20%20if%20n%20%3D%3D%200%3A%0A%20%20%20%20%20%20%20%20return%201%0A...%0A%0A

 Share with team!


In [ ]:
example_code = (
    "def calculate_factorial(n):\n"
    "    if n == 0:\n"
    "        return 1\n"
    "    return n * calculate_factorial(n-1)"
)

final_code_with_docstring, status_message = generate_docstring(example_code, style="google")
print(final_code_with_docstring)
print(status_message)

def calculate_factorial(n):
"""Executes calculate_factorial operation with conditional logic performing mathematical calculations and returns the computed result.

Args:
    n (Any): Parameter for controlling n behavior.

Returns:
    Any: The result of the computation.
"""
    if n == 0:
        return 1
    return n * calculate_factorial(n-1)

✅ GOOGLE Docstring Generated!


In [ ]:
share_link_whatsapp = generate_whatsapp_link()
print(share_link_whatsapp)

✅ WhatsApp Link:
https://wa.me/?text=%F0%9F%93%9A%20%2ADoc-Genie%20Documentation%2A%0A%0A%2ADocumentation%20%231%2A%0AStyle%3A%20GOOGLE%0ACode%3A%20def%20calculate_factorial%28n%29%3A%0A%20%20%20%20if%20n%20%3D%3D%200%3A%0A%20%20%20%20%20%20%20%20return%201%0A...%0A%0A

 Share with team!


In [ ]:
print("\n✅ Doc-Genie v2.0 Ready! Launching Gradio Demo...\n")
demo.launch(share=True, show_error=True, show_api=False)

/tmp/ipykernel_580/3816797853.py:2: DeprecationWarning: The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].
  demo.launch(share=True, show_error=True, show_api=False)



✅ Doc-Genie v2.0 Ready! Launching Gradio Demo...

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7e484410081d38ed1e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
